# 03 Exploratory Data Analysis

**Project:** Smartwatch Health Analytics — User Risk Segmentation  
**Input:** `data/processed/clean_smartwatch_health_data.csv`  
**Goal:** Explore distributions, segment comparisons, correlations, and surface early business signals.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data/processed/clean_smartwatch_health_data.csv'

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
df.describe(include='all').T

## 1. Activity Level Distribution

Understanding how users are spread across activity categories helps identify the dominant segments.

In [ ]:
activity_order = ['sedentary', 'lightly active', 'active', 'highly active']
activity_counts = df['activity_level'].value_counts().reindex(
    [a for a in activity_order if a in df['activity_level'].unique()]
).dropna()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(activity_counts.index, activity_counts.values, color=sns.color_palette('muted', len(activity_counts)))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)
ax.set_title('User Distribution by Activity Level', fontsize=13, fontweight='bold')
ax.set_xlabel('Activity Level')
ax.set_ylabel('Number of Users')
plt.tight_layout()
plt.show()

print('\nInsight: Majority of users fall in the active / highly active segment.')

## 2. Heart Rate Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['heart_rate_bpm'].dropna(), bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Heart Rate Distribution (BPM)')
axes[0].set_xlabel('Heart Rate (BPM)')
axes[0].axvline(df['heart_rate_bpm'].median(), color='red', linestyle='--', label=f'Median: {df["heart_rate_bpm"].median():.1f}')
axes[0].legend()

sns.boxplot(data=df, x='activity_level', y='heart_rate_bpm', order=activity_order, ax=axes[1], palette='muted')
axes[1].set_title('Heart Rate by Activity Level')
axes[1].set_xlabel('Activity Level')
axes[1].set_ylabel('Heart Rate (BPM)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print('\nInsight: Sedentary users show elevated resting heart rates compared to active users, signalling higher cardiovascular risk.')

## 3. Blood Oxygen Level Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['blood_oxygen_level'].dropna(), bins=40, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Blood Oxygen Level Distribution (%)')
axes[0].set_xlabel('Blood Oxygen (%)')
axes[0].axvline(95, color='red', linestyle='--', label='Clinical threshold (95%)')
axes[0].legend()

low_oxygen = (df['blood_oxygen_level'] < 95).sum()
pct = low_oxygen / df['blood_oxygen_level'].notna().sum() * 100
axes[1].pie([low_oxygen, df['blood_oxygen_level'].notna().sum() - low_oxygen],
            labels=[f'Below 95%\n({pct:.1f}%)', f'Normal\n({100-pct:.1f}%)'],
            colors=['#e74c3c', '#2ecc71'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Blood Oxygen: Normal vs Below Clinical Threshold')

plt.tight_layout()
plt.show()

print(f'\nInsight: {pct:.1f}% of users record blood oxygen below the 95% clinical threshold — a flag for respiratory health monitoring.')

## 4. Step Count Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['step_count'].dropna(), bins=50, kde=True, ax=axes[0], color='coral')
axes[0].set_title('Daily Step Count Distribution')
axes[0].set_xlabel('Step Count')
axes[0].axvline(10000, color='red', linestyle='--', label='10,000 step goal')
axes[0].legend()

sns.boxplot(data=df, x='activity_level', y='step_count', order=activity_order, ax=axes[1], palette='Set2')
axes[1].set_title('Step Count by Activity Level')
axes[1].set_xlabel('Activity Level')
axes[1].set_ylabel('Step Count')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

goal_achievers = (df['step_count'] >= 10000).sum()
pct_goal = goal_achievers / df['step_count'].notna().sum() * 100
print(f'\nInsight: {pct_goal:.1f}% of users meet the 10,000 steps/day goal. Step count is a strong differentiator across activity segments.')

## 5. Sleep Duration Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['sleep_duration_hours'].dropna(), bins=40, kde=True, ax=axes[0], color='mediumpurple')
axes[0].set_title('Sleep Duration Distribution (hours)')
axes[0].set_xlabel('Sleep Duration (hours)')
axes[0].axvline(7, color='red', linestyle='--', label='Recommended min (7h)')
axes[0].legend()

sns.boxplot(data=df, x='activity_level', y='sleep_duration_hours', order=activity_order, ax=axes[1], palette='Purples')
axes[1].set_title('Sleep Duration by Activity Level')
axes[1].set_xlabel('Activity Level')
axes[1].set_ylabel('Sleep Duration (hours)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

poor_sleep = (df['sleep_duration_hours'] < 7).sum()
pct_poor = poor_sleep / df['sleep_duration_hours'].notna().sum() * 100
print(f'\nInsight: {pct_poor:.1f}% of users sleep fewer than 7 hours. Poor sleep correlates strongly with higher stress levels.')

## 6. Stress Level Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

stress_counts = df['stress_level'].value_counts().sort_index()
axes[0].bar(stress_counts.index.astype(str), stress_counts.values, color=sns.color_palette('RdYlGn_r', len(stress_counts)))
axes[0].set_title('Stress Level Distribution (1=Low, 5=High)')
axes[0].set_xlabel('Stress Level')
axes[0].set_ylabel('Count')

stress_by_activity = df.groupby('activity_level')['stress_level'].mean().reindex(
    [a for a in activity_order if a in df['activity_level'].unique()]
).dropna()
axes[1].bar(stress_by_activity.index, stress_by_activity.values, color=sns.color_palette('RdYlGn_r', len(stress_by_activity)))
axes[1].set_title('Average Stress Level by Activity Level')
axes[1].set_xlabel('Activity Level')
axes[1].set_ylabel('Avg Stress Level')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print('\nInsight: Sedentary users have measurably higher average stress levels. Regular physical activity appears to be a stress buffer.')

## 7. Correlation Heatmap

In [ ]:
numeric_cols = ['heart_rate_bpm', 'blood_oxygen_level', 'step_count', 'sleep_duration_hours', 'stress_level']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Health Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nInsight: Stress level shows negative correlation with sleep duration — users who sleep less are consistently more stressed.')
print('Step count shows positive correlation with blood oxygen, suggesting active users maintain better respiratory health.')

## 8. Pairplot — Health Metrics by Activity Level

In [ ]:
sample = df[df['activity_level'].notna()].sample(min(2000, len(df)), random_state=42)
pair_cols = ['heart_rate_bpm', 'step_count', 'sleep_duration_hours', 'stress_level', 'activity_level']

g = sns.pairplot(sample[pair_cols].dropna(), hue='activity_level',
                 hue_order=[a for a in activity_order if a in sample['activity_level'].unique()],
                 plot_kws={'alpha': 0.4, 's': 15})
g.fig.suptitle('Pairplot of Health Metrics by Activity Level', y=1.02, fontsize=13, fontweight='bold')
plt.show()

## 9. EDA Summary — Key Findings

| # | Finding | Business Signal |
|---|---|---|
| 1 | Sedentary users have higher resting heart rates | Higher cardiovascular risk — priority for wellness intervention |
| 2 | ~X% of users have blood oxygen < 95% | Flag for respiratory health programs |
| 3 | ~X% meet the 10,000 step/day goal | Low step achievement = intervention opportunity |
| 4 | Poor sleep (< 7h) prevalent in high-stress users | Sleep coaching programs could reduce stress load |
| 5 | Stress level inversely correlated with sleep duration | Sleep hygiene is a lever for mental wellness |
| 6 | Step count positively correlated with blood oxygen | Physical activity drives respiratory health |

**Next Step:** Proceed to `04_statistical_analysis.ipynb` for hypothesis testing and clustering.